In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. 파일 경로 설정 (내 맥북 로컬 경로)
parquet_path = "volume/shared/2026w09.zstd.parquet"

# 2. Parquet 파일 로드 및 데이터 확인
df = pd.read_parquet(parquet_path)
print(f"Total Rows: {len(df)}")
display(df.head()) # 데이터 구조 미리보기

In [ ]:
from glob import glob

In [ ]:
files = sorted(glob("volume/shared/*.zstd.parquet"), reverse=True)
files

In [ ]:
# 3. StarRocks 연결 (MySQL 호환 프로토콜 활용)
SR_USER = "root"
SR_PASS = ""
SR_HOST = "172.16.1.65" 
SR_PORT = "9030" # FE Query Port
SR_DB = "huq"

# 연결 엔진 생성
connection_str = f"mysql+pymysql://{SR_USER}:{SR_PASS}@{SR_HOST}:{SR_PORT}/{SR_DB}"
engine = create_engine(
    connection_str,
    connect_args={"connect_timeout": 600, "read_timeout": 600, "write_timeout": 600},
    pool_pre_ping=True, # 연결이 살아있는지 확인 후 쿼리 실행
    pool_recycle=3600
)

In [ ]:
create_table_sql = """
CREATE TABLE bronze (
    -- Key 컬럼들을 맨 앞으로 배치
    app_id_hash             VARCHAR(255),
    `timestamp`             DATETIME,
    
    -- 나머지 일반 컬럼들
    country                 VARCHAR(10),
    city                    VARCHAR(100),
    device_board            VARCHAR(100),
    device_brand            VARCHAR(100),
    device_carrier_code     VARCHAR(50),
    device_carrier_name     VARCHAR(100),
    device_device           VARCHAR(100),
    device_display_hardware VARCHAR(255),
    device_display_height   BIGINT,
    device_display_width    BIGINT,
    device_fingerprint      VARCHAR(500),
    device_hardware         VARCHAR(100),
    device_host             VARCHAR(100),
    device_iid_hash         VARCHAR(255),
    device_language         VARCHAR(10),
    device_locale           VARCHAR(10),
    device_manufacturer     VARCHAR(100),
    device_model            VARCHAR(100),
    device_product          VARCHAR(100),
    device_sim_code         VARCHAR(50),
    device_tac              VARCHAR(50),
    device_total_memory     BIGINT
) ENGINE=OLAP
DUPLICATE KEY(app_id_hash, `timestamp`)
DISTRIBUTED BY HASH(app_id_hash) BUCKETS 16;
"""

# DB명을 제외한 기본 연결 주소 (관리용)
ADMIN_CONN = f"mysql+pymysql://{SR_USER}:{SR_PASS}@{SR_HOST}:{SR_PORT}/"
engine_admin = create_engine(ADMIN_CONN)

# 데이터베이스 생성
with engine_admin.begin() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS huq"))
    print("✅ 데이터베이스 'huq' 준비 완료")

with engine.begin() as conn:
    conn.execute(text(create_table_sql))
    print("✅ 테이블 'bronze' 생성 완료")

In [ ]:
def change_type(df: pd.DataFrame) -> pd.DataFrame:
    int_columns = ["device_display_height", "device_display_width", "device_total_memory"]

    for x in df.columns:
        if x in ["timestamp"]:
            continue
        
        if x in int_columns:
            df[x] = df[x].fillna(0).astype(int)
            continue
        
        df[x] = df[x].fillna(0).astype(str)
            
    df['timestamp'] = pd.to_datetime(df['timestamp']).dt.strftime('%Y-%m-%d %H:%M:%S')
    return df

In [ ]:
import time
from tqdm.auto import tqdm
from sqlalchemy import create_engine, text

# 1. 타임아웃 설정을 대폭 늘린 새로운 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{SR_USER}:{SR_PASS}@{SR_HOST}:9030/{SR_DB}",
    connect_args={
        "connect_timeout": 600, 
        "read_timeout": 600, 
        "write_timeout": 600
    },
    pool_pre_ping=True, # 실행 전 연결 확인
    pool_recycle=3600
)

In [ ]:
for x in files:
    print(f"filename: {x}")
    
    df = pd.read_parquet(x)
    print(f"Total Rows: {len(df):,}")
    
    df = change_type(df=df)
        
    # 2. 설정
    chunk_size = 2000  # 5000에서 2000으로 줄여서 패킷 크기 감소 (안정성)
    total_rows = len(df)
    columns_list = df.columns.tolist()

    columns_str = ", ".join([f"`{col}`" for col in columns_list])
    placeholders = ", ".join([f":{col}" for col in columns_list])
    insert_query = text(f"INSERT INTO bronze ({columns_str}) VALUES ({placeholders})")

    # 3. 에러 난 지점(50,000)부터 다시 시작하려면 start_idx를 조정하세요.
    # 처음부터라면 0, 이어서 하려면 50000
    start_idx = 0     

    with engine.connect() as conn:
        for i in tqdm(range(start_idx, total_rows, chunk_size)):
            chunk = df.iloc[i : i + chunk_size]
            
            # 모든 nan 값을 None으로 치환 (ProgrammingError 방지)
            data_dict = [
                {k: (None if pd.isna(v) else v) for k, v in record.items()} 
                for record in chunk.to_dict(orient='records')
            ]
            
            try:
                # 개별 청크마다 트랜잭션 수행
                with conn.begin():
                    conn.execute(insert_query, data_dict)
                
            except Exception as inner_e:
                print(f"⚠️ {i}행 인근에서 일시적 에러 발생, 30초 후 재연결 시도: {inner_e}")
                time.sleep(30)
                
                conn = engine.connect() # 연결 새로 맺기
                
                # 해당 청크 다시 시도
                with conn.begin():
                    conn.execute(insert_query, data_dict)